# **Análisis exploratorio del MITSUI&CO. Commodity Prediction Challenge**

## **Inspección y descripción inicial del dataset**

El propósito de esta primera sección es comprender **qué datos están disponibles y cómo se organizan**. La inspección se limita a describir la estructura de los archivos, las funciones de sus variables y las agrupaciones sugeridas por su nomenclatura. En este punto no se diagnostican valores faltantes, duplicados o atípicos, ni se toman decisiones de limpieza.

## **¿Por qué este dataset utiliza varios archivos?**

En un problema tabular sencillo es común encontrar un solo CSV que contiene predictores y una variable objetivo. Este reto es distinto porque separa la información según la función que cumple dentro del problema predictivo:

- **`train.csv`** contiene las observaciones históricas que pueden utilizarse como variables predictoras: precios, volúmenes, tipos de cambio y otras mediciones de instrumentos financieros.
- **`train_labels.csv`** contiene los resultados que se busca predecir. En lugar de una sola variable objetivo, incluye 424 targets, desde `target_0` hasta `target_423`.
- **`target_pairs.csv`** funciona como metadata de los targets. Explica qué instrumento o diferencia entre instrumentos origina cada target y cuál es su rezago.
- **`test.csv`** reproduce la estructura de los predictores para la etapa de predicción e incorpora `is_scored`, una variable auxiliar que indica qué filas participan en la evaluación de Kaggle.

La separación evita mezclar datos observables, respuestas futuras y metadata. También permite que Kaggle entregue predictores sin revelar anticipadamente las respuestas utilizadas para evaluar un modelo.

## **Preparación de la inspección**

Las siguientes celdas localizan la raíz del proyecto y cargan los archivos directamente desde `data/raw/`. 

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 100)


def encontrar_raiz_proyecto() -> Path:
    """Encuentra la raiz que contiene data/raw desde el directorio actual."""
    directorio_actual = Path.cwd().resolve()
    for candidato in (directorio_actual, *directorio_actual.parents):
        if (candidato / "data" / "raw").is_dir():
            return candidato
    raise FileNotFoundError("No se encontro la carpeta data/raw del proyecto.")


RAIZ_PROYECTO = encontrar_raiz_proyecto()
DIR_RAW = RAIZ_PROYECTO / "data" / "raw"

RUTAS = {
    "train": DIR_RAW / "train.csv",
    "train_labels": DIR_RAW / "train_labels.csv",
    "target_pairs": DIR_RAW / "target_pairs.csv",
    "test": DIR_RAW / "test.csv",
}

archivos_no_encontrados = [str(ruta) for ruta in RUTAS.values() if not ruta.is_file()]
if archivos_no_encontrados:
    raise FileNotFoundError(f"Faltan archivos: {archivos_no_encontrados}")

RAIZ_PROYECTO

PosixPath('/Users/bryan/Documents/DataScience/Proyecto_2_EDA_Data_Science')

In [2]:
datos = {
    nombre: pd.read_csv(ruta, low_memory=False)
    for nombre, ruta in RUTAS.items()
}

train = datos["train"]
train_labels = datos["train_labels"]
target_pairs = datos["target_pairs"]
test = datos["test"]

print("Archivos cargados correctamente:", ", ".join(RUTAS))

Archivos cargados correctamente: train, train_labels, target_pairs, test


## **Inventario de archivos**

El inventario resume el tamaño y la cobertura del identificador temporal de cada archivo. `target_pairs.csv` no posee `date_id` porque describe targets, no observaciones diarias.

In [3]:
FUNCIONES_ARCHIVOS = {
    "train": "Predictores historicos",
    "train_labels": "Variables objetivo historicas",
    "target_pairs": "Metadata para interpretar los targets",
    "test": "Predictores para la etapa de evaluacion",
}


def resumir_archivo(nombre: str, df: pd.DataFrame) -> dict:
    tiene_fecha = "date_id" in df.columns
    return {
        "archivo": RUTAS[nombre].name,
        "funcion": FUNCIONES_ARCHIVOS[nombre],
        "filas": df.shape[0],
        "columnas": df.shape[1],
        "date_id_inicial": df["date_id"].min() if tiene_fecha else pd.NA,
        "date_id_final": df["date_id"].max() if tiene_fecha else pd.NA,
    }


inventario = pd.DataFrame(
    [resumir_archivo(nombre, df) for nombre, df in datos.items()]
).set_index("archivo")
inventario

,funcion,filas,columnas,date_id_inicial,date_id_final
archivo,,,,,
train.csv,Predictores historicos,1961,558,0,1960
train_labels.csv,Variables objetivo historicas,1961,425,0,1960
target_pairs.csv,Metadata para interpretar los targets,424,3,<NA>,<NA>
test.csv,Predictores para la etapa de evaluacion,134,559,1827,1960


El inventario permite observar dos correspondencias importantes. Primero, `train.csv` y `train_labels.csv` poseen la misma cantidad de filas y comparten la cobertura de `date_id`, lo que permite vincular cada observación histórica con sus respuestas. Segundo, las 424 filas de `target_pairs.csv` corresponden a las 424 columnas objetivo de `train_labels.csv`.

## **Tipos de datos inferidos**

Debido a la alta dimensionalidad, se resume cuántas columnas fueron interpretadas con cada tipo en lugar de imprimir una lista de cientos de variables. Esta observación sirve para comprender la estructura; el tipado definitivo se evaluará más adelante.

In [4]:
registros_tipos = []
for nombre, df in datos.items():
    for tipo, cantidad in df.dtypes.astype(str).value_counts().items():
        registros_tipos.append(
            {"archivo": RUTAS[nombre].name, "tipo_inferido": tipo, "columnas": cantidad}
        )

resumen_tipos = pd.DataFrame(registros_tipos).sort_values(
    ["archivo", "columnas"], ascending=[True, False]
)
resumen_tipos

,archivo,tipo_inferido,columnas
4,target_pairs.csv,str,2
5,target_pairs.csv,int64,1
6,test.csv,float64,557
7,test.csv,int64,1
8,test.csv,bool,1
0,train.csv,float64,557
1,train.csv,int64,1
2,train_labels.csv,float64,424
3,train_labels.csv,int64,1


La estructura inferida muestra que `train.csv` y `train_labels.csv` están formados por variables numéricas, además de `date_id`. No se observan predictores categóricos explícitos. Las variables textuales aparecen en `target_pairs.csv`, donde `target` y `pair` cumplen una función descriptiva, mientras que `lag` es numérica. En `test.csv`, `is_scored` es una variable booleana auxiliar.

## **Función de las variables**

Las columnas pueden organizarse por el papel que cumplen dentro del reto. Esta clasificación separa los identificadores de las variables que un modelo podría utilizar, las respuestas que intentaría predecir y la metadata necesaria para interpretarlas.

In [5]:
variables_predictoras = [columna for columna in train.columns if columna != "date_id"]
variables_objetivo = [columna for columna in train_labels.columns if columna.startswith("target_")]
variables_metadata = target_pairs.columns.tolist()
variables_auxiliares_test = [columna for columna in test.columns if columna == "is_scored"]

resumen_funciones = pd.DataFrame(
    [
        {
            "funcion": "Identificador temporal",
            "cantidad": 1,
            "ejemplos": "date_id",
        },
        {
            "funcion": "Variables predictoras",
            "cantidad": len(variables_predictoras),
            "ejemplos": ", ".join(variables_predictoras[:3]),
        },
        {
            "funcion": "Variables objetivo",
            "cantidad": len(variables_objetivo),
            "ejemplos": f"{variables_objetivo[0]}, ..., {variables_objetivo[-1]}",
        },
        {
            "funcion": "Columnas de metadata de targets",
            "cantidad": len(variables_metadata),
            "ejemplos": ", ".join(variables_metadata),
        },
        {
            "funcion": "Variables auxiliares de test",
            "cantidad": len(variables_auxiliares_test),
            "ejemplos": ", ".join(variables_auxiliares_test),
        },
    ]
)
resumen_funciones

,funcion,cantidad,ejemplos
0,Identificador temporal,1,date_id
1,Variables predictoras,557,"LME_AH_Close, LME_CA_Close, LME_PB_Close"
2,Variables objetivo,424,"target_0, ..., target_423"
3,Columnas de metadata de targets,3,"target, lag, pair"
4,Variables auxiliares de test,1,is_scored


### **Descripción de la metadata de targets y la variable auxiliar**

Las tres columnas de `target_pairs.csv` permiten interpretar las variables objetivo sin formar parte de las observaciones históricas:

- **`target`**: contiene el nombre de la variable objetivo a la que se refiere cada fila, por ejemplo, `target_0`. Este nombre permite vincular la metadata con la columna correspondiente de `train_labels.csv`.
- **`lag`**: indica la cantidad de días de rezago utilizada para construir el retorno futuro del target. En este dataset toma valores del 1 al 4, por lo que distingue el horizonte temporal asociado con cada objetivo.
- **`pair`**: identifica el instrumento o los instrumentos utilizados para calcular el target. Cuando presenta un solo instrumento, el objetivo representa su retorno; cuando aparecen dos instrumentos separados por ` - `, representa la diferencia entre sus retornos.

Por otro lado, **`is_scored`** pertenece exclusivamente a `test.csv`. Es una variable booleana que indica si una fila se incluye en el cálculo de la métrica de evaluación de Kaggle: `True` señala una fila evaluada y `False` una fila proporcionada como contexto, pero no puntuada. Por tanto, cumple una función auxiliar de evaluación y no representa una característica financiera del mercado.

## **Agrupación de predictores por mercado**

Los nombres de las columnas incorporan un prefijo que identifica su procedencia. Esto permite resumir las 557 variables predictoras sin describirlas manualmente una por una.

In [6]:
def clasificar_mercado(nombre_variable: str) -> str:
    """Clasifica una variable segun el prefijo definido por la competencia."""
    if nombre_variable.startswith("LME_"):
        return "London Metal Exchange (LME)"
    if nombre_variable.startswith("JPX_"):
        return "Japan Exchange Group (JPX)"
    if nombre_variable.startswith("US_Stock_"):
        return "Acciones de Estados Unidos"
    if nombre_variable.startswith("FX_"):
        return "Mercado de divisas (FX)"
    return "Sin clasificar"


resumen_mercados = (
    pd.Series(variables_predictoras, name="variable")
    .map(clasificar_mercado)
    .value_counts()
    .rename_axis("mercado")
    .reset_index(name="cantidad_variables")
)
resumen_mercados

,mercado,cantidad_variables
0,Acciones de Estados Unidos,475
1,Japan Exchange Group (JPX),40
2,Mercado de divisas (FX),38
3,London Metal Exchange (LME),4


### **Agrupación complementaria por medición**

Además del mercado, la terminación de los nombres permite reconocer si una columna representa apertura, máximo, mínimo, cierre, volumen, precio de liquidación, interés abierto, una medición ajustada de acciones o un tipo de cambio.

In [7]:
def clasificar_medicion(nombre_variable: str) -> str:
    """Resume la medicion indicada por la nomenclatura de una variable."""
    nombre_normalizado = nombre_variable.lower()
    if nombre_normalizado.startswith("fx_"):
        return "Tipo de cambio"
    sufijos = {
        "_adj_open": "Precio ajustado de apertura",
        "_adj_high": "Precio máximo ajustado",
        "_adj_low": "Precio mínimo ajustado",
        "_adj_close": "Precio ajustado de cierre",
        "_adj_volume": "Volumen ajustado",
        "_settlement_price": "Precio de liquidación",
        "_open_interest": "Interés abierto",
        "_open": "Apertura",
        "_high": "Máximo",
        "_low": "Mínimo",
        "_close": "Cierre",
        "_volume": "Volumen",
    }
    for sufijo, medicion in sufijos.items():
        if nombre_normalizado.endswith(sufijo):
            return medicion
    return "Otra medicion"


resumen_mediciones = (
    pd.Series(variables_predictoras, name="variable")
    .map(clasificar_medicion)
    .value_counts()
    .rename_axis("medicion")
    .reset_index(name="cantidad_variables")
)
resumen_mediciones

,medicion,cantidad_variables
0,Precio ajustado de apertura,95
1,Precio máximo ajustado,95
2,Precio mínimo ajustado,95
3,Precio ajustado de cierre,95
4,Volumen ajustado,95
5,Tipo de cambio,38
6,Cierre,10
7,Apertura,6
8,Máximo,6
9,Mínimo,6


### **Descripción de las mediciones**

Las mediciones ajustadas corresponden a las acciones de Estados Unidos y buscan mantener la comparabilidad histórica ante eventos corporativos. Las mediciones sin ajustar describen principalmente las sesiones e instrumentos de LME y JPX, mientras que los tipos de cambio pertenecen al mercado de divisas:

- **Precio ajustado de apertura**: precio de apertura de la sesión corregido para reflejar eventos corporativos, como divisiones de acciones, según el ajuste aplicado por la fuente.
- **Precio máximo ajustado**: mayor precio alcanzado durante la sesión después de aplicar el factor de ajuste correspondiente.
- **Precio mínimo ajustado**: menor precio registrado durante la sesión después de aplicar el mismo criterio de ajuste.
- **Precio ajustado de cierre**: precio de cierre corregido para conservar la comparabilidad histórica ante eventos como divisiones de acciones y, según la fuente, distribuciones de dividendos.
- **Volumen ajustado**: cantidad negociada corregida para que cambios como las divisiones de acciones no produzcan saltos puramente mecánicos en la serie histórica.
- **Tipo de cambio**: valor de una moneda expresado en unidades de otra, identificado mediante el par de divisas correspondiente.
- **Cierre**: último precio o precio de referencia registrado al finalizar la sesión de negociación.
- **Apertura**: primer precio o precio de referencia registrado al iniciar la sesión de negociación.
- **Máximo**: precio más alto alcanzado por el instrumento durante la sesión.
- **Mínimo**: precio más bajo alcanzado por el instrumento durante la sesión.
- **Volumen**: cantidad de unidades o contratos negociados durante la sesión.
- **Interés abierto**: número de contratos de futuros que permanecen vigentes y todavía no han sido cerrados o liquidados al finalizar la sesión.
- **Precio de liquidación**: precio de referencia establecido al cierre por la bolsa para valorar posiciones y calcular ganancias, pérdidas o requerimientos de margen.

## **Organización de las variables objetivo**

Los nombres `target_0` a `target_423` no describen por sí solos qué se predice. Esa información se encuentra en `target_pairs.csv`:

- `target` identifica la columna correspondiente de `train_labels.csv`.
- `lag` indica el horizonte o rezago utilizado para construir el retorno futuro.
- `pair` contiene un instrumento individual o una diferencia entre dos instrumentos.

Por esta razón, `target_pairs.csv` debe entenderse como un diccionario estructural de los targets y no como un conjunto adicional de observaciones.

In [8]:
def separar_instrumentos(expresion: str) -> list[str]:
    """Separa el instrumento individual o los dos componentes de un target."""
    return [parte.strip() for parte in expresion.split(" - ")]


estructura_targets = target_pairs.assign(
    cantidad_instrumentos=target_pairs["pair"].map(lambda valor: len(separar_instrumentos(valor))),
)
estructura_targets["tipo_target"] = estructura_targets["cantidad_instrumentos"].map(
    {1: "Retorno de un instrumento", 2: "Diferencia entre dos instrumentos"}
)
estructura_targets["mercados_involucrados"] = estructura_targets["pair"].map(
    lambda valor: " + ".join(
        sorted({clasificar_mercado(instrumento) for instrumento in separar_instrumentos(valor)})
    )
)

resumen_targets_por_lag = pd.crosstab(
    estructura_targets["lag"],
    estructura_targets["tipo_target"],
    margins=True,
    margins_name="Total",
)
resumen_targets_por_lag

tipo_target,Diferencia entre dos instrumentos,Retorno de un instrumento,Total
lag,,,
1,105,1,106
2,105,1,106
3,105,1,106
4,105,1,106
Total,420,4,424


La distribución es uniforme entre los cuatro rezagos: cada `lag` contiene 106 targets, de los cuales 105 representan diferencias entre dos instrumentos y uno corresponde al retorno de un instrumento individual. En total, 420 de los 424 objetivos se construyen como diferencias, mientras que únicamente cuatro representan retornos individuales.

In [9]:
resumen_targets_por_mercado = (
    estructura_targets["mercados_involucrados"]
    .value_counts()
    .rename_axis("mercados_involucrados")
    .reset_index(name="cantidad_targets")
)
resumen_targets_por_mercado

,mercados_involucrados,cantidad_targets
0,Acciones de Estados Unidos + London Metal Exchange (LME),181
1,Acciones de Estados Unidos + Japan Exchange Group (JPX),87
2,London Metal Exchange (LME) + Mercado de divisas (FX),86
3,Japan Exchange Group (JPX) + Mercado de divisas (FX),46
4,Japan Exchange Group (JPX) + London Metal Exchange (LME),12
5,London Metal Exchange (LME),8
6,Acciones de Estados Unidos,2
7,Mercado de divisas (FX),2
